# 01 — Umami Analytics Ingestion

Validate the Umami Cloud REST API integration:
- Connect with API key
- Fetch website stats, pageviews, metrics, events
- Parse into our `Insights` model
- Save to local state (mock S3)

In [ ]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
import os

load_dotenv('../.env', override=True)

UMAMI_API_KEY = os.getenv('UMAMI_API_KEY')
UMAMI_WEBSITE_ID = os.getenv('UMAMI_WEBSITE_ID', 'e41ae7d9-a536-426d-b40e-f2488b11bf95')

print(f'API Key loaded: {"yes" if UMAMI_API_KEY else "NO — create .env from .env.example"}')
print(f'Website ID: {UMAMI_WEBSITE_ID}')

## 1. Connect to Umami Cloud API

Umami Cloud base URL: `https://api.umami.is/v1`  
Auth: `x-umami-api-key` header  
Rate limit: 50 calls / 15 seconds

In [ ]:
from agent.umami_client import UmamiClient, ms_timestamp

umami = UmamiClient(api_key=UMAMI_API_KEY, website_id=UMAMI_WEBSITE_ID)

# Test connection: get active users
active = umami.get_active()
print(f'Active users right now: {active}')

## 2. Fetch Website Stats (last 7 days)

In [ ]:
start = ms_timestamp(days_ago=7)
end = ms_timestamp(days_ago=0)

stats = umami.get_stats(start_at=start, end_at=end)
print('Website stats (7 days):')
for k, v in stats.items():
    if k != 'comparison':
        print(f'  {k}: {v}')

## 3. Top Pages

In [ ]:
top_pages = umami.get_metrics(start_at=start, end_at=end, metric_type='path', limit=15)
print('Top pages:')
for page in top_pages:
    print(f'  {page["y"]:>5} visitors — {page["x"]}')

## 4. Top Referrers

In [ ]:
referrers = umami.get_metrics(start_at=start, end_at=end, metric_type='referrer', limit=15)
print('Top referrers:')
for ref in referrers:
    print(f'  {ref["y"]:>5} visitors — {ref["x"]}')

## 5. Events (tracked funnels)

In [ ]:
events = umami.get_metrics(start_at=start, end_at=end, metric_type='event', limit=30)
print('Tracked events:')
for event in events:
    print(f'  {event["y"]:>5}x — {event["x"]}')

## 6. Pageviews Over Time

In [ ]:
pageviews = umami.get_pageviews(start_at=start, end_at=end, unit='day')
print('Daily pageviews:')
for pv in pageviews.get('pageviews', []):
    print(f'  {pv["x"][:10]} — {pv["y"]} views')

print('\nDaily sessions:')
for s in pageviews.get('sessions', []):
    print(f'  {s["x"][:10]} — {s["y"]} sessions')

In [ ]:
from datetime import datetime
from agent.models import Insights, WebsiteAnalytics

insights = Insights(
    website_analytics=WebsiteAnalytics(
        pageviews=stats.get('pageviews', 0),
        visitors=stats.get('visitors', 0),
        visits=stats.get('visits', 0),
        bounces=stats.get('bounces', 0),
        totaltime=stats.get('totaltime', 0),
        top_pages=top_pages[:10],
        top_referrers=referrers[:10],
        top_events=events[:10],
    ),
    last_analysis=datetime.now(),
)

print(f'Insights parsed: {insights.website_analytics.visitors} visitors, '
      f'{len(insights.website_analytics.top_pages)} top pages')


## 7. Save to S3 State

In [ ]:
from agent.storage import S3Storage

store = S3Storage(
    bucket=os.getenv('S3_BUCKET'),
    prefix=os.getenv('S3_STATE_PREFIX', 'growth-agent/'),
    access_key=os.getenv('SCW_ACCESS_KEY'),
    secret_key=os.getenv('SCW_SECRET_KEY'),
)
store.write('insights.json', insights)

# Verify round-trip
loaded = store.read('insights.json')
print(f'Saved and loaded insights.json ({len(str(loaded))} chars)')
print(f'Top page: {loaded["website_analytics"]["top_pages"][0] if loaded["website_analytics"]["top_pages"] else "none"}')

In [ ]:
umami.close()
print('Done — Umami ingestion validated.')